# IEEE-CIS Fraud Detection: 비용 민감형 사기탐지 모델 개발
## Day 7. 확률 보정(Calibration) + Threshold Optimization

**목표**
- LightGBM(튜닝), RF(기본값) 두 최종 후보에 CalibratedClassifierCV 보정 적용
- 보정 전후 PR-AUC + Calibration 비교하여 최종 모델 선정
- 비용함수(FP/FN) 기반 Threshold Optimization

**진행 순서**
1. 데이터 + 변수 세트 불러오기
2. LightGBM best params 불러오기 (day6_tuning_results.pkl)
3. LightGBM(튜닝) + RF(기본값) 재학습
4. CalibratedClassifierCV 보정 적용
5. 보정 전후 PR-AUC + Calibration Gap 비교
6. 최종 모델 선정
7. Threshold Optimization

### 1. 라이브러리 임포트 및 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import sys
sys.path.append("..")

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', 100)

# 데이터
df = pd.read_parquet("../data/processed/day3_final.parquet")

# LightGBM best params
with open("../data/processed/day6_tuning_results.pkl", 'rb') as f:
    tuning_results = pickle.load(f)

best_params_lgbm = tuning_results['best_params']['LightGBM']

# 변수 세트 재정의 (Day 4/6과 동일)
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT', 'UID', 'UID_v2']
target = 'isFraud'
groups = df['UID']

categorical_cols = df.drop(columns=exclude_cols).select_dtypes(include='category').columns.tolist()
numeric_cols = df.drop(columns=exclude_cols).select_dtypes(include=[np.number]).columns.tolist()
binary_cols = [c for c in numeric_cols if df[c].nunique() <= 2]
scale_cols = [c for c in numeric_cols if c not in binary_cols]

X = df.drop(columns=exclude_cols + [target])
y = df[target]

print(f"shape: {df.shape}")
print(f"LightGBM best params: {best_params_lgbm}")

c:\Users\seonu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shape: (590540, 208)
LightGBM best params: {'learning_rate': 0.1563037874293778, 'num_leaves': 165, 'min_child_samples': 32, 'subsample': 0.9248534606596772, 'colsample_bytree': 0.6651160762107319, 'reg_lambda': 0.00025077983643406634}


### 2. 사전 함수 재정의

Day 4/6에서 사용한 타겟 인코딩 함수를 재정의합니다.
커널 재부팅으로 인해 이전 노트북의 함수가 사라졌으므로 재정의가 필요합니다.

In [2]:
def target_encode(train_df, valid_df, cols, target_col, smoothing=20):
    global_mean = train_df[target_col].mean()
    encoded_train = train_df.copy()
    encoded_valid = valid_df.copy()
    for col in cols:
        encoded_train[col] = encoded_train[col].astype(str)
        encoded_valid[col] = encoded_valid[col].astype(str)
        stats = train_df.assign(**{col: train_df[col].astype(str)}).groupby(col)[target_col].agg(['mean', 'count'])
        smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        encoded_train[col] = encoded_train[col].map(smoothed).fillna(global_mean)
        encoded_valid[col] = encoded_valid[col].map(smoothed).fillna(global_mean)
    return encoded_train, encoded_valid

print("함수 정의 완료")

함수 정의 완료


### 3. CalibratedClassifierCV 개념 및 적용 방식

**CalibratedClassifierCV란**
모델이 출력하는 예측 확률을 실제 사기 발생 확률에 가깝게 보정하는 방법입니다.
내부적으로 CV를 사용하여 보정 모델을 학습하므로, 보정 과정에서 데이터 누수가 없습니다.

**보정 방법 두 가지**
- Platt Scaling (method='sigmoid'): 예측 확률에 Logistic Regression을 추가로 피팅. 간단하고 빠름
- Isotonic Regression (method='isotonic'): 단조 증가 함수로 보정. 데이터가 많을 때 더 유연하게 작동

**선택: Isotonic Regression**
우리 데이터는 59만 건으로 충분히 크므로 Isotonic Regression이 더 유연한 보정을 제공함.

**적용 방식**
- cv=5로 설정하여 5-fold CV 내부에서 보정 모델을 학습
- StratifiedGroupKFold가 아닌 일반 cv=5를 사용하는 이유:
  CalibratedClassifierCV는 보정 자체의 안정성을 위한 내부 CV이며,
  UID 그룹 분리는 외부 평가(Day 4 run_cv)에서 이미 적용했음

### 참고: Isotonic Regression을 선택한 이유

**Platt Scaling vs Isotonic Regression 비교**

| 구분 | Platt Scaling | Isotonic Regression |
|---|---|---|
| 보정 방식 | sigmoid(A × 원래확률 + B), S자 함수로 보정 | 단조 증가 조건만 만족하면 어떤 형태도 가능 |
| 유연성 | 낮음 (S자 형태로만 보정 가능) | 높음 (복잡한 왜곡 패턴도 보정 가능) |
| 필요 데이터 | 적어도 안정적 | 많을수록 안정적 (과적합 위험 있음) |
| 적합한 상황 | 왜곡이 단순하거나 데이터가 적을 때 | 왜곡이 크고 복잡하며 데이터가 충분할 때 |

**우리가 Isotonic Regression을 선택한 이유**

Day 5 Calibration 분석에서 확인한 Gap이 꽤 컸음:
- RF: 평균 Gap 0.1366 (전 구간 과소신)
- LightGBM: 평균 Gap 0.3130 (전 구간 과신, 중간 구간에서 0.5 이상)

이렇게 왜곡 패턴이 복잡하고 Gap이 큰 경우, Platt Scaling의 단순한 S자 함수로는 보정이 충분하지 않을 수 있음. Isotonic Regression의 유연한 보정이 더 효과적임.

또한 우리 데이터는 59만 건으로 충분히 크기 때문에, Isotonic Regression의 "데이터가 많아야 과적합 없이 안정적으로 작동한다"는 조건도 문제없이 충족함.

**면접 한 줄 설명**
"Calibration Gap이 크고 복잡한 패턴을 보여서 단순 S자 함수인 Platt Scaling보다 더 유연하게 보정할 수 있는 Isotonic Regression을 선택했고, 59만 건의 충분한 데이터가 있어 과적합 위험도 낮다고 판단했습니다."

In [3]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

imbalance_ratio = (y == 0).sum() / (y == 1).sum()

# 최종 후보 2개 모델 정의
model_lgbm = LGBMClassifier(
    **best_params_lgbm,
    n_estimators=5000,
    scale_pos_weight=imbalance_ratio,
    random_state=42,
    verbosity=-1,
)

model_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)

# CalibratedClassifierCV 적용
calibrated_lgbm = CalibratedClassifierCV(model_lgbm, method='isotonic', cv=5)
calibrated_rf   = CalibratedClassifierCV(model_rf,   method='isotonic', cv=5)

print("모델 정의 완료")
print(f"LightGBM best params: {best_params_lgbm}")
print(f"클래스 불균형 비율: 1:{imbalance_ratio:.1f}")

모델 정의 완료
LightGBM best params: {'learning_rate': 0.1563037874293778, 'num_leaves': 165, 'min_child_samples': 32, 'subsample': 0.9248534606596772, 'colsample_bytree': 0.6651160762107319, 'reg_lambda': 0.00025077983643406634}
클래스 불균형 비율: 1:27.6


### 4. 보정 전후 비교를 위한 OOF 평가

보정 전후 성능을 공정하게 비교하기 위해,
StratifiedGroupKFold 5-fold OOF 방식으로 보정된 모델의 PR-AUC와 Calibration Gap을 측정합니다.

보정 전 결과(Day 4/5)와 동일한 조건에서 비교합니다.
- LightGBM 보정 전: PR-AUC 0.4860 (기본값), 0.5101 (튜닝 후)
- RF 보정 전: PR-AUC 0.4972 (기본값)

In [4]:
def evaluate_calibrated_model(model, model_name, use_category=False):
    """보정된 모델의 OOF PR-AUC + Calibration Gap 측정"""
    sgkf = StratifiedGroupKFold(n_splits=5)
    oof_preds  = np.zeros(len(df))
    oof_labels = np.zeros(len(df))
    pr_aucs    = []

    for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups)):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx]
        y_val   = y.iloc[val_idx]

        if not use_category:
            temp_train = pd.concat([X_train[categorical_cols], y_train], axis=1)
            temp_val   = X_val[categorical_cols].copy()
            temp_train_enc, temp_val_enc = target_encode(
                temp_train, temp_val, categorical_cols, target
            )
            X_train[categorical_cols] = temp_train_enc[categorical_cols].values
            X_val[categorical_cols]   = temp_val_enc[categorical_cols].values

        scaler = StandardScaler()
        X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
        X_val[scale_cols]   = scaler.transform(X_val[scale_cols])

        if use_category:
            for col in categorical_cols:
                X_train[col] = X_train[col].astype('category')
                X_val[col]   = X_val[col].astype('category')
        else:
            for col in categorical_cols:
                X_train[col] = X_train[col].astype(float)
                X_val[col]   = X_val[col].astype(float)

        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_val)[:, 1]

        oof_preds[val_idx]  = y_pred
        oof_labels[val_idx] = y_val.values

        pr_auc = average_precision_score(y_val, y_pred)
        pr_aucs.append(pr_auc)
        print(f"  [Fold {fold+1}] PR-AUC: {pr_auc:.4f}")

    # Calibration Gap 계산
    prob_true, prob_pred = calibration_curve(
        oof_labels, oof_preds, n_bins=10, strategy='uniform'
    )
    gap = np.abs(prob_true - prob_pred)
    brier = brier_score_loss(oof_labels, oof_preds)

    print(f"\n  [{model_name}] 평균 PR-AUC: {np.mean(pr_aucs):.4f} (±{np.std(pr_aucs):.4f})")
    print(f"  [{model_name}] 평균 |Gap|: {gap.mean():.4f} | Brier Score: {brier:.4f}")

    return {
        'model_name':  model_name,
        'pr_auc_mean': np.mean(pr_aucs),
        'pr_auc_std':  np.std(pr_aucs),
        'mean_gap':    gap.mean(),
        'brier':       brier,
        'oof_preds':   oof_preds,
        'oof_labels':  oof_labels,
    }


print("=== LightGBM (튜닝 + 보정) ===")
result_lgbm = evaluate_calibrated_model(calibrated_lgbm, 'LightGBM(보정)', use_category=True)

print("\n=== Random Forest (기본값 + 보정) ===")
result_rf = evaluate_calibrated_model(calibrated_rf, 'RF(보정)', use_category=False)

=== LightGBM (튜닝 + 보정) ===
  [Fold 1] PR-AUC: 0.5472
  [Fold 2] PR-AUC: 0.5537
  [Fold 3] PR-AUC: 0.5581
  [Fold 4] PR-AUC: 0.5070
  [Fold 5] PR-AUC: 0.5923

  [LightGBM(보정)] 평균 PR-AUC: 0.5517 (±0.0272)
  [LightGBM(보정)] 평균 |Gap|: 0.1447 | Brier Score: 0.0241

=== Random Forest (기본값 + 보정) ===
  [Fold 1] PR-AUC: 0.5157
  [Fold 2] PR-AUC: 0.5125
  [Fold 3] PR-AUC: 0.5154
  [Fold 4] PR-AUC: 0.4504
  [Fold 5] PR-AUC: 0.5561

  [RF(보정)] 평균 PR-AUC: 0.5100 (±0.0339)
  [RF(보정)] 평균 |Gap|: 0.0507 | Brier Score: 0.0231


In [5]:
import pickle, os

calibration_results = {
    'lgbm_calibrated': {
        'pr_auc_mean': result_lgbm['pr_auc_mean'],
        'pr_auc_std': result_lgbm['pr_auc_std'],
        'mean_gap': result_lgbm['mean_gap'],
        'brier': result_lgbm['brier'],
        'oof_preds': result_lgbm['oof_preds'],
        'oof_labels': result_lgbm['oof_labels'],
    },
    'rf_calibrated': {
        'pr_auc_mean': result_rf['pr_auc_mean'],
        'pr_auc_std': result_rf['pr_auc_std'],
        'mean_gap': result_rf['mean_gap'],
        'brier': result_rf['brier'],
        'oof_preds': result_rf['oof_preds'],
        'oof_labels': result_rf['oof_labels'],
    }
}

with open("../data/processed/day7_calibration_results.pkl", 'wb') as f:
    pickle.dump(calibration_results, f)

print("저장 완료")
print(f"LightGBM(보정): PR-AUC={result_lgbm['pr_auc_mean']:.4f}, Gap={result_lgbm['mean_gap']:.4f}")
print(f"RF(보정): PR-AUC={result_rf['pr_auc_mean']:.4f}, Gap={result_rf['mean_gap']:.4f}")

저장 완료
LightGBM(보정): PR-AUC=0.5517, Gap=0.1447
RF(보정): PR-AUC=0.5100, Gap=0.0507
